# Phase 1 — Embedding Extraction with BiomedCLIP

This notebook runs **phase 1** of the SAE pipeline:
it extracts visual and text embeddings from the IU X-Ray dataset
using the `chuhac/BiomedCLIP-vit-bert-hf` model.

**Input:**
- `data/iu_xray/images/images_normalized/` — 7 470 PNG chest X-rays
- `data/iu_xray/reports/indiana_reports.csv` — clinical reports

**Output:**
- `embeddings/visual_embeddings.pt` — tensor `(N_img, 512)`
- `embeddings/text_embeddings.pt`  — tensor `(N_rep, 512)`

**Device:** MPS (Apple Silicon) → CUDA → CPU fallback

## 0. Setup & Configuration

In [1]:
import sys
from pathlib import Path

import torch

# Traverse up to the project root
PROJECT_ROOT = Path.cwd()
while not (PROJECT_ROOT / 'src').exists() and PROJECT_ROOT != PROJECT_ROOT.parent:
    PROJECT_ROOT = PROJECT_ROOT.parent

sys.path.insert(0, str(PROJECT_ROOT / 'src'))
sys.path.insert(0, str(PROJECT_ROOT / 'datasets'))

# Select compute device (MPS > CUDA > CPU)
if torch.backends.mps.is_available():
    DEVICE = 'mps'
elif torch.cuda.is_available():
    DEVICE = 'cuda'
else:
    DEVICE = 'cpu'

print(f'Project root : {PROJECT_ROOT}')
print(f'PyTorch      : {torch.__version__}')
print(f'Device       : {DEVICE}')

Project root : /home/marcantoniolopez/Documenti/github/xai-project-5
PyTorch      : 2.12.0+cu130
Device       : cuda


In [2]:
import config

# Actual paths after extracting the Kaggle zip
DATASET_ROOT = PROJECT_ROOT / 'data' / 'iu_xray' / 'chest-xrays-indiana-university'

vlm_cfg = config.VLMConfig(
    device=DEVICE,
    image_dir=str(DATASET_ROOT / 'images' / 'images_normalized'),
    reports_dir=str(DATASET_ROOT),  # CSV is directly in the extracted root
    output_dir=str(PROJECT_ROOT / 'embeddings'),
    batch_size=32,   # keep <64 for MPS stability
    num_workers=0,   # 0 = main process (required on macOS)
)

print('=== VLMConfig ===')
print(f'  model       : {vlm_cfg.model_name}')
print(f'  device      : {vlm_cfg.device}')
print(f'  batch_size  : {vlm_cfg.batch_size}')
print(f'  image_dir   : {vlm_cfg.image_dir}')
print(f'  reports_dir : {vlm_cfg.reports_dir}')
print(f'  visual_out  : {vlm_cfg.visual_output_path}')
print(f'  text_out    : {vlm_cfg.text_output_path}')

=== VLMConfig ===
  model       : chuhac/BiomedCLIP-vit-bert-hf
  device      : cuda
  batch_size  : 32
  image_dir   : /home/marcantoniolopez/Documenti/github/xai-project-5/data/iu_xray/chest-xrays-indiana-university/images/images_normalized
  reports_dir : /home/marcantoniolopez/Documenti/github/xai-project-5/data/iu_xray/chest-xrays-indiana-university
  visual_out  : /home/marcantoniolopez/Documenti/github/xai-project-5/embeddings/visual_embeddings.pt
  text_out    : /home/marcantoniolopez/Documenti/github/xai-project-5/embeddings/text_embeddings.pt


In [3]:
images_dir = DATASET_ROOT / 'images' / 'images_normalized'
reports_csv = DATASET_ROOT / 'indiana_reports.csv'

print('=== IU X-Ray staging ===')
print(f'Images directory : {images_dir}')
print(f'Reports CSV      : {reports_csv}')

if not images_dir.exists():
    raise FileNotFoundError(f'Missing images directory: {images_dir}')

if not reports_csv.exists():
    raise FileNotFoundError(f'Missing report CSV: {reports_csv}')

n_images = len(list(images_dir.glob('*.png')))
print(f'PNG files found  : {n_images}')
print('Dataset staged correctly.')

=== IU X-Ray staging ===
Images directory : /home/marcantoniolopez/Documenti/github/xai-project-5/data/iu_xray/chest-xrays-indiana-university/images/images_normalized
Reports CSV      : /home/marcantoniolopez/Documenti/github/xai-project-5/data/iu_xray/chest-xrays-indiana-university/indiana_reports.csv
PNG files found  : 7470
Dataset staged correctly.


## 1. Data Verification

In [4]:
from iu_xray import IUXrayImageDataset, IUXrayTextDataset

image_dataset = IUXrayImageDataset(Path(vlm_cfg.image_dir))
text_dataset  = IUXrayTextDataset(Path(vlm_cfg.reports_dir))

print(f'Images found  : {len(image_dataset)}')
print(f'Reports found : {len(text_dataset)}')

# Visual sample
img, path = image_dataset[0]
print(f'\nSample image: {Path(path).name} — mode={img.mode}, size={img.size}')

# Text sample
text, uid = text_dataset[0]
print(f'Sample text : uid={uid}')
print(f'  → {text[:120]}...')

Images found  : 7470
Reports found : 3851

Sample image: 1000_IM-0003-1001.dcm.png — mode=RGB, size=(2048, 2496)
Sample text : uid=1
  → Findings: The cardiac silhouette and mediastinum size are within normal limits. There is no pulmonary edema. There is no...


## 2. Load BiomedCLIP

In [5]:
from transformers import AutoModel, AutoProcessor
from transformers.models.clip.configuration_clip import CLIPConfig
import transformers.models.clip.modeling_clip as _clip_mod

# ── Compatibility patches for BiomedCLIP custom code + newer transformers ──

# 1) CLIPConfig now rejects positional args; the cached BiomedCLIP config passes them.
_orig_clip_init = CLIPConfig.__init__
def _clip_init_compat(self, *args, **kwargs):
    if args:
        names = ['text_config', 'vision_config', 'projection_dim', 'logit_scale_init_value']
        for name, val in zip(names, args):
            if name not in kwargs:
                kwargs[name] = val
    _orig_clip_init(self, **kwargs)
CLIPConfig.__init__ = _clip_init_compat

# 2) BiomedCLIP uses `from modeling_clip import *` but __all__ restricts exports.
if hasattr(_clip_mod, '__all__'):
    del _clip_mod.__all__

print(f'Loading model: {vlm_cfg.model_name} ...')
processor = AutoProcessor.from_pretrained(vlm_cfg.processor_name, trust_remote_code=True)
model     = AutoModel.from_pretrained(vlm_cfg.model_name, trust_remote_code=True)

# 3) BiomedCLIP text tower checks config.is_decoder (BERT-style) but CLIPTextConfig lacks it.
if not hasattr(model.text_model.config, 'is_decoder'):
    model.text_model.config.is_decoder = False

# 4) Fix corrupted position_ids buffer (PyTorch 2.12 + persistent=False bug).
import torch
max_pos = model.text_model.config.max_position_embeddings
model.text_model.embeddings.position_ids = torch.arange(max_pos).unsqueeze(0)

model = model.to(DEVICE)
model.eval()

print(f'Model on {DEVICE} — total parameters: {sum(p.numel() for p in model.parameters()):,}')

/home/marcantoniolopez/Documenti/github/xai-project-5/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Loading model: chuhac/BiomedCLIP-vit-bert-hf ...


[transformers] Model config: bos_token_id must be `None` or an integer within the vocabulary (between 0 and 30521), got 49406. This may result in unexpected behavior.
[transformers] Model config: eos_token_id must be `None` or an integer within the vocabulary (between 0 and 30521), got 49407. This may result in unexpected behavior.
[transformers] `BiomedCLIPProcessor` defines `image_processor_class = 'CLIPImageProcessor'`, which is deprecated. Register the correct mapping in `AutoImageProcessor` instead.
Loading weights: 100%|██████████| 399/399 [00:00<00:00, 14492.93it/s]


Model on cuda — total parameters: 195,902,721


## 3. Visual Embedding Extraction

In [6]:
from extract_embeddings import extract_visual_embeddings

extract_visual_embeddings(model, processor, image_dataset, vlm_cfg)

# Verify output
visual_emb = torch.load(vlm_cfg.visual_output_path, map_location='cpu', weights_only=True)
print(f'\nvisual_embeddings.pt : {visual_emb.shape}  dtype={visual_emb.dtype}')
print(f'  mean norm          : {visual_emb.norm(dim=-1).mean():.4f}  (expected ~1.0)')


Found 7470 images. Starting extraction...


Images Processing: 100%|██████████| 234/234 [05:23<00:00,  1.38s/it]

Images Embedding Extraction completed. Saving on /home/marcantoniolopez/Documenti/github/xai-project-5/embeddings/visual_embeddings.pt.

visual_embeddings.pt : torch.Size([7470, 512])  dtype=torch.float32
  mean norm          : 1.0000  (expected ~1.0)


## 4. Text Embedding Extraction

In [7]:
from extract_embeddings import extract_text_embeddings

extract_text_embeddings(model, processor, text_dataset, vlm_cfg)

# Verify output
text_emb = torch.load(vlm_cfg.text_output_path, map_location='cpu', weights_only=True)
print(f'\ntext_embeddings.pt : {text_emb.shape}  dtype={text_emb.dtype}')
print(f'  mean norm        : {text_emb.norm(dim=-1).mean():.4f}  (expected ~1.0)')


Found 3851 reports. Starting extraction...


Reports Processing: 100%|██████████| 121/121 [00:14<00:00,  8.26it/s]

Reports Embedding Extraction completed. Saving on /home/marcantoniolopez/Documenti/github/xai-project-5/embeddings/text_embeddings.pt.

text_embeddings.pt : torch.Size([3851, 512])  dtype=torch.float32
  mean norm        : 1.0000  (expected ~1.0)


## 5. Embedding Quality Check

In [8]:
import numpy as np

# Load from disk
v_emb = torch.load(vlm_cfg.visual_output_path, map_location='cpu', weights_only=True)
t_emb = torch.load(vlm_cfg.text_output_path, map_location='cpu', weights_only=True)

print('═══ Embedding Statistics ═══\n')
print(f'Visual embeddings : {v_emb.shape}  ({v_emb.dtype})')
print(f'Text embeddings   : {t_emb.shape}  ({t_emb.dtype})')

# Norms (should be ~1.0 for L2-normalized)
v_norms = v_emb.norm(dim=-1)
t_norms = t_emb.norm(dim=-1)
print(f'\n── Norms ──')
print(f'Visual: mean={v_norms.mean():.6f}, std={v_norms.std():.6f}, min={v_norms.min():.6f}, max={v_norms.max():.6f}')
print(f'Text  : mean={t_norms.mean():.6f}, std={t_norms.std():.6f}, min={t_norms.min():.6f}, max={t_norms.max():.6f}')

# Distribution of values
print(f'\n── Value Distribution ──')
print(f'Visual: mean={v_emb.mean():.6f}, std={v_emb.std():.6f}, min={v_emb.min():.6f}, max={v_emb.max():.6f}')
print(f'Text  : mean={t_emb.mean():.6f}, std={t_emb.std():.6f}, min={t_emb.min():.6f}, max={t_emb.max():.6f}')

# Sparsity (fraction of near-zero activations)
v_sparse = (v_emb.abs() < 0.01).float().mean()
t_sparse = (t_emb.abs() < 0.01).float().mean()
print(f'\n── Sparsity (|x| < 0.01) ──')
print(f'Visual: {v_sparse:.4f} ({v_sparse*100:.1f}%)')
print(f'Text  : {t_sparse:.4f} ({t_sparse*100:.1f}%)')

# Cosine similarity within each modality (sample)
def avg_pairwise_cosine(emb, n_sample=500):
    idx = torch.randperm(len(emb))[:n_sample]
    sample = emb[idx]
    sim = sample @ sample.T
    mask = ~torch.eye(n_sample, dtype=bool)
    return sim[mask].mean().item(), sim[mask].std().item()

v_cos_mean, v_cos_std = avg_pairwise_cosine(v_emb)
t_cos_mean, t_cos_std = avg_pairwise_cosine(t_emb)
print(f'\n── Avg Pairwise Cosine Similarity (sample 500) ──')
print(f'Visual-Visual: {v_cos_mean:.4f} ± {v_cos_std:.4f}')
print(f'Text-Text    : {t_cos_mean:.4f} ± {t_cos_std:.4f}')

# Cross-modal similarity
cross_sim = (v_emb[:500] @ t_emb[:500].T).mean().item()
print(f'Cross-modal  : {cross_sim:.4f}')

# File sizes
import os
v_size = os.path.getsize(vlm_cfg.visual_output_path) / (1024**2)
t_size = os.path.getsize(vlm_cfg.text_output_path) / (1024**2)
print(f'\n── File Sizes ──')
print(f'visual_embeddings.pt : {v_size:.2f} MB')
print(f'text_embeddings.pt   : {t_size:.2f} MB')
print(f'Total                : {v_size + t_size:.2f} MB')

═══ Embedding Statistics ═══

Visual embeddings : torch.Size([7470, 512])  (torch.float32)
Text embeddings   : torch.Size([3851, 512])  (torch.float32)

── Norms ──
Visual: mean=1.000000, std=0.000000, min=1.000000, max=1.000000
Text  : mean=1.000000, std=0.000000, min=1.000000, max=1.000000

── Value Distribution ──
Visual: mean=-0.000478, std=0.044192, min=-0.341892, max=0.365037
Text  : mean=0.000665, std=0.044189, min=-0.235362, max=0.550903

── Sparsity (|x| < 0.01) ──
Visual: 0.2136 (21.4%)
Text  : 0.2379 (23.8%)

── Avg Pairwise Cosine Similarity (sample 500) ──
Visual-Visual: 0.7940 ± 0.1124
Text-Text    : 0.8018 ± 0.0759
Cross-modal  : 0.3532

── File Sizes ──
visual_embeddings.pt : 14.59 MB
text_embeddings.pt   : 7.52 MB
Total                : 22.11 MB
